# Google Keyword Planner ideas analysis
Reproducible profile of the first 100 visible ideas supplied on 2026-08-13. These are demand estimates, not DataCost performance.

In [1]:
from pathlib import Path
import os
import pandas as pd

source = Path(os.environ.get('DATACOST_GOOGLE_KEYWORD_IDEAS_FILE', 'google-keyword-ideas.txt')).expanduser()
lines = [line.strip() for line in source.read_text(encoding='utf-8').splitlines()]
visible = lines[lines.index('Keyword ideas') + 1:lines.index('Show rows')]
assert len(visible) % 8 == 0
rows = [visible[index:index + 8] for index in range(0, len(visible), 8)]
ideas = pd.DataFrame(rows, columns=['keyword', 'avg_monthly_searches', 'three_month_change', 'yoy_change', 'competition', 'ad_impression_share', 'low_bid', 'high_bid'])
ideas['avg_monthly_searches'] = ideas['avg_monthly_searches'].str.replace(',', '', regex=False).astype(int)
{'visible_rows': len(ideas), 'unique_keywords': ideas.keyword.str.lower().nunique(), 'combined_searches': int(ideas.avg_monthly_searches.sum()), 'available_ideas_shown_in_source': 1639}

{'visible_rows': 100,
 'unique_keywords': 100,
 'combined_searches': 101440,
 'available_ideas_shown_in_source': 1639}

In [2]:
ambiguous = ideas[ideas.keyword.str.lower().eq('vodacom deals')]
{'ambiguous_vodacom_deals': int(ambiguous.avg_monthly_searches.iloc[0]), 'share_of_visible_pct': round(ambiguous.avg_monthly_searches.iloc[0] / ideas.avg_monthly_searches.sum() * 100, 1), 'visible_total_excluding_ambiguous_term': int(ideas.avg_monthly_searches.sum() - ambiguous.avg_monthly_searches.iloc[0])}

{'ambiguous_vodacom_deals': 60500,
 'share_of_visible_pct': np.float64(59.6),
 'visible_total_excluding_ambiguous_term': 40940}

In [3]:
themes = {
    'unlimited_or_uncapped': r'unlimited|uncapped',
    'month_to_month_or_contract': r'month to month|contract',
    'cheap_best_or_affordable': r'cheap|best|affordable',
    'prepaid': r'prepaid',
    'lte': r'\blte\b',
}
theme_summary = pd.DataFrame([{'theme': theme, 'rows': int(ideas.keyword.str.contains(pattern, case=False, regex=True).sum()), 'searches': int(ideas.loc[ideas.keyword.str.contains(pattern, case=False, regex=True), 'avg_monthly_searches'].sum())} for theme, pattern in themes.items()])
theme_summary.sort_values('searches', ascending=False)

,theme,rows,searches
0,unlimited_or_uncapped,27,14880
1,month_to_month_or_contract,17,6690
4,lte,9,4810
2,cheap_best_or_affordable,22,3290
3,prepaid,11,1100


## Guardrails
Only rows 1–100 of 1,639 were supplied. Themes overlap and cannot be summed. Planner targeting settings are not visible; paid competition and GBP bids are not organic difficulty or confirmed South African traffic value.